# 🧹 Session 1: Understanding & Cleaning Data
**Data Science Workshop · Part 1 of 2 · ~60 minutes**

> **📦 The scenario:** You just joined **ShopSmart**, an online store, as a data analyst.
> On day one your manager sends you last year's order export and says:
> *"Can we trust this data? Please clean it so we can analyze it next week."*

### 🗺️ Agenda
| ⏱ | Section | You will learn |
|---|---|---|
| 0–5 | 1. What is data? | Data types, datasets, tidy data |
| 5–15 | 2. Reading data | CSV, Excel, JSON, Parquet, and common traps |
| 15–25 | 3. Data types & first look | `info`, `describe`, `value_counts`, dtypes |
| 25–50 | 4. Data quality & cleaning | Duplicates, types, categories, invalid values, missing data |
| 50–60 | 5. Challenge | Validate your data against reference data |


| Icon | Meaning |
|---|---|
| 📖 | Concept / theory |
| 💻 | Run this code and look at the output |
| 🧪 | Exercise: your turn! |
| ⚠️ | Common pitfall |
| 💡 | Tip / best practice |
| 🚀 | Stretch goal (for advanced participants, optional) |

**How to run a cell:** click it and press **Shift + Enter**. To start fresh, use **Kernel → Restart Kernel and Run All Cells**.

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

# Works whether the notebook is in the workshop root or in /solutions
DATA = Path("data") if Path("data").exists() else Path("..") / "data"
print("pandas", pd.__version__, "| data folder:", DATA.resolve())

### 🐍 Python & pandas in 2 minutes *(optional, skip if you know Python)*
A **DataFrame** is a table, like an Excel sheet. A **Series** is one column of that table.

In [ ]:
# list = ordered collection, dict = key -> value, function = reusable logic
prices = [19.99, 5.50, 120.00]
order = {"id": "ORD-1", "qty": 2, "price": 19.99}

def order_total(qty, price, discount=0):
    return qty * price * (1 - discount)

print("Total:", order_total(order["qty"], order["price"], discount=0.1))

mini = pd.DataFrame({"product": ["pen", "book", "lamp"], "price": prices})
mini

In [ ]:
print(mini["price"].mean())          # a column (Series) + an aggregation
mini[mini["price"] > 10]             # filtering rows with a condition (boolean mask)

---
## 1. 📖 What is data? *(⏱ 5 min)*

**Data** = recorded observations about the world. A **dataset** = a collection of data organized for a purpose.

| Kind | Description | Examples |
|---|---|---|
| **Structured** | Rows & columns with a fixed schema | SQL tables, CSV, Excel |
| **Semi-structured** | Has structure, but flexible/nested | JSON, XML, logs |
| **Unstructured** | No predefined model | Images, audio, free text, PDFs |

### Vocabulary
- **Row** = observation = record = sample (one order)
- **Column** = variable = feature = attribute (e.g. `unit_price`)
- **Value** = a single cell

### 💡 Tidy data (Hadley Wickham)
1. Each **variable** is a column
2. Each **observation** is a row
3. Each **value** is a cell

### The data lifecycle
`Collect → Store → Clean → Explore → Model/Analyze → Communicate`

> It's often said that data professionals spend **60–80 % of their time cleaning data**. Today you'll see why.

---
## 2. 📥 Reading data *(⏱ 10 min)*

| Format | pandas function | Notes |
|---|---|---|
| CSV / TXT | `pd.read_csv()` | Most common. Watch the **separator**, **encoding**, **decimal** |
| Excel | `pd.read_excel()` | Multiple **sheets**, junk header rows, merged cells |
| JSON | `pd.read_json()` / `pd.json_normalize()` | Often **nested** |
| Parquet | `pd.read_parquet()` | Columnar, compressed, **keeps data types**. Great for analytics |
| SQL | `pd.read_sql()` | Needs a DB connection |
| Web / API | `requests` + `pd.json_normalize()` | Pagination, auth |

### 💻 2.1 CSV: our main dataset

In [ ]:
raw = pd.read_csv(DATA / "orders_raw.csv")
print(raw.shape)   # (rows, columns)
raw.head()

💡 Always look at more than the first rows: use `.tail()` and `.sample()` too. Problems hide in the middle!

In [ ]:
raw.sample(5, random_state=1)

### ⚠️ 2.2 Not every CSV is "comma separated"
Our European office sent `orders_eu.csv`. Let's read it the default way:

In [ ]:
try:
    eu = pd.read_csv(DATA / "orders_eu.csv")
    display(eu.head())
except UnicodeDecodeError as e:
    print("💥", type(e).__name__, "-", e)

When reading fails, **look at the raw bytes** of the file:

In [ ]:
with open(DATA / "orders_eu.csv", "rb") as f:
    print(f.read(250))

We can see:
- `;` is the **separator**
- `,` is the **decimal** mark (`590,07`)
- the `\xc1` byte is a non-UTF-8 character → the encoding is **latin-1**

In [ ]:
eu = pd.read_csv(DATA / "orders_eu.csv", sep=";", decimal=",", encoding="latin-1")
print(eu.dtypes)
eu.head()

### 💻 2.3 JSON: nested data

In [ ]:
customers_nested = pd.read_json(DATA / "customers.json")
customers_nested.head(3)      # 'profile' contains dictionaries 😬

In [ ]:
import json

with open(DATA / "customers.json") as f:
    records = json.load(f)
print(records[0])

customers = pd.json_normalize(records)   # flattens nested keys -> profile.age, profile.city
customers.head()

### 💻 2.4 Excel: multiple sheets and junk rows

In [ ]:
xls = pd.ExcelFile(DATA / "products.xlsx")
print("Sheets:", xls.sheet_names)
pd.read_excel(xls, sheet_name="catalog").head()   # the header is wrong!

In [ ]:
products = pd.read_excel(xls, sheet_name="catalog", skiprows=2)
products.head()

### 📖 2.5 Parquet
CSV stores everything as **text**, so types are lost and must be re-guessed on every read.
**Parquet** stores data **by column**, **compressed**, and **with types**. We'll save our clean data as Parquet at the end.

### 🧪 Exercise 2.1: Read only what you need

Read **only** the columns `product_id`, `category`, `list_price` from the `catalog` sheet (hint: `usecols=`),
then count how many products exist per category.

In [ ]:
# your code here

### 🧪 Exercise 2.2 🚀: Clean while reading

Read only the first 200 rows of `orders_raw.csv` (`nrows=`) and tell pandas that `"unknown"`, `"-"` and `"?"` mean missing (`na_values=`).
What is the dtype of `customer_age` now?

In [ ]:
# your code here

---
## 3. 🔎 Data types & first look *(⏱ 10 min)*

### 📖 Statistical data types
| Type | Sub-type | In our data | Ideal pandas dtype |
|---|---|---|---|
| **Numerical** | Continuous | `unit_price`, `total_amount` | `float64` |
| | Discrete | `quantity`, `delivery_days` | `int64` |
| **Categorical** | Nominal (no order) | `city`, `category`, `payment_method` | `category` / `str` |
| | Ordinal (ordered) | `customer_rating` 1–5, membership Basic < Silver < Gold | ordered `category` |
| **Datetime** | | `order_date` | `datetime64` |
| **Boolean** | | `returned` | `bool` |
| **Identifier** | | `order_id`, `customer_id` | `str` (never do math on IDs!) |
| **Free text** | | `notes` | `str` |

The type decides **which operations make sense**: you can average prices but not cities, and you can count cities but not sum them.

In [ ]:
raw.info()

🕵️ **What looks wrong?** `unit_price`, `customer_age`, `order_date` and `returned` are stored as **text** (`str`/`object`).
That means something inside those columns is not a clean number, date, or boolean. Let's find it:

In [ ]:
as_number = pd.to_numeric(raw["unit_price"], errors="coerce")   # unparseable -> NaN
raw.loc[as_number.isna(), "unit_price"].head(10)

### 💻 Summary statistics: your first data-quality radar

In [ ]:
raw.describe()

🤔 Look at the **min** and **max** rows:
- `quantity` has a negative minimum → impossible
- `discount` max is way above 1 (100 %) → was it typed as a percent?
- `delivery_days` max is ~200 days → typo or a real problem?
- `session_minutes` max > 10 hours → suspicious

In [ ]:
raw.describe(exclude="number")

In [ ]:
raw.nunique().sort_values()      # 1 unique value = useless column

In [ ]:
raw["city"].value_counts()       # same city, many spellings!

In [ ]:
raw["returned"].value_counts()   # 8 ways of saying yes/no

💡 **The `category` dtype** saves memory and is faster for columns with few unique values:

In [ ]:
before = raw["city"].memory_usage(deep=True)
after = raw["city"].astype("category").memory_usage(deep=True)
print(f"str: {before / 1024:.0f} KB  →  category: {after / 1024:.0f} KB")

### 🧪 Exercise 3.1: What is hiding in customer_age?

Find the values in `customer_age` that are **not missing** but also **not numbers**, and count them.

In [ ]:
# hint: pd.to_numeric(..., errors="coerce"), .isna(), .notna()

### 🧪 Exercise 3.2: Spot the problems

Use `value_counts(dropna=False)` on `category` and `payment_method`. Write down every problem you can see.

In [ ]:
# your code here

---
## 4. 🧽 Data quality & cleaning *(⏱ 25 min)*

### 📖 Data-quality dimensions
| Dimension | Question | Example in our data |
|---|---|---|
| **Uniqueness** | Is each record there only once? | Duplicate orders |
| **Validity** | Does the value obey the rules/format? | Age = 999, price = "$1,234" |
| **Consistency** | Is the same thing written the same way? | "NY", "NYC", "new york" |
| **Completeness** | Is anything missing? | Missing ratings, ages |
| **Accuracy** | Is it actually true? | Total doesn't match qty × price |
| **Timeliness** | Is it up to date? | Old exports |

### 💡 Golden rules
1. **Never modify the raw data.** Work on a copy.
2. **Log every step** so others can reproduce it.
3. **Check before and after** each step.
4. When unsure, **ask the business** instead of guessing.

In [ ]:
df = raw.copy()
cleaning_log = []

def log_step(step, rows_before):
    cleaning_log.append({"step": step, "rows_before": rows_before, "rows_after": len(df)})
    print(f"✅ {step}: {rows_before:,} → {len(df):,} rows")

### 4.1 Uniqueness: duplicates

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate order_ids :", df["order_id"].duplicated().sum())

In [ ]:
n = len(df)
df = df.drop_duplicates()
log_step("Drop exact duplicate rows", n)

Some `order_id`s still appear twice, but the rows are **not identical** (*near-duplicates*). Let's look:

In [ ]:
dup_ids = df.loc[df["order_id"].duplicated(keep=False), "order_id"]
df[df["order_id"].isin(dup_ids)].sort_values("order_id").head(6)

Strategy: keep the **most complete** version of each order.

In [ ]:
n = len(df)
df["_n_missing"] = df.isna().sum(axis=1)
df = (
    df.sort_values("_n_missing", kind="stable")
      .drop_duplicates(subset="order_id", keep="first")
      .drop(columns="_n_missing")
      .sort_index()
)
log_step("Drop duplicate order_ids (keep most complete)", n)

### 🧪 Exercise 4.1: Customers

How many **unique customers** are there? Which customer placed the **most orders**?

In [ ]:
# your code here

### 4.2 Remove useless columns

In [ ]:
print("currency values:", df["currency"].unique().tolist())
print(f"notes missing: {df['notes'].isna().mean():.1%}")
df = df.drop(columns=["currency", "notes"])

### 4.3 Validity: fix the data types

In [ ]:
# Price: remove everything that is not a digit or a dot ("$1,234.50 USD" -> "1234.50")
df["unit_price"] = pd.to_numeric(df["unit_price"].astype(str).str.replace(r"[^0-9.]", "", regex=True))

# Age: anything that's not a number becomes NaN
df["customer_age"] = pd.to_numeric(df["customer_age"], errors="coerce")

df[["unit_price", "customer_age"]].dtypes

In [ ]:
df["order_date"].dropna().sample(8, random_state=3).tolist()   # 4 different date formats!

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")
df["order_date"].describe()

⚠️ Is `03/04/2024` March 4th or April 3rd? `format="mixed"` assumes US (month first) unless you pass `dayfirst=True`. **Always confirm the convention with the data owner.**

In [ ]:
returned_map = {"yes": True, "y": True, "1": True, "true": True,
                "no": False, "n": False, "0": False, "false": False}
df["returned"] = df["returned"].astype(str).str.strip().str.lower().map(returned_map)
print(df["returned"].value_counts(dropna=False))
df["returned"] = df["returned"].astype(bool)

### 4.4 Consistency: one name per thing

In [ ]:
df["city"] = df["city"].str.strip().str.title()     # trim spaces, consistent casing
df["city"].value_counts()

In [ ]:
city_fix = {"Ny": "New York", "Nyc": "New York", "La": "Los Angeles", "Seatle": "Seattle"}
df["city"] = df["city"].replace(city_fix)

category_fix = {"Electronic": "Electronics", "Home And Kitchen": "Home & Kitchen",
                "Home&Kitchen": "Home & Kitchen", "Clothes": "Clothing",
                "Book": "Books", "Sport": "Sports"}
df["category"] = df["category"].str.strip().str.title().replace(category_fix)

df["payment_method"] = df["payment_method"].replace({"?": np.nan})   # fake missing -> real missing

print(df["city"].value_counts(), end="\n\n")
print(df["category"].value_counts())

### 🧪 Exercise 4.2: Double-check

Check `shipping_method` and `payment_method`. Are there still inconsistent values?

In [ ]:
# your code here

### 4.5 Validity: impossible values (business rules)
These rules come from **domain knowledge**, not statistics: a customer can't be 999 years old, and a quantity can't be negative.

In [ ]:
bad_age = df["customer_age"].notna() & ~df["customer_age"].between(18, 100)
print("Ages outside 18-100:", bad_age.sum(), sorted(df.loc[bad_age, "customer_age"].unique().tolist()))
print("Quantity <= 0       :", (df["quantity"] <= 0).sum())
print("Discount > 1        :", (df["discount"] > 1).sum(), "e.g.", df.loc[df["discount"] > 1, "discount"].unique()[:5].tolist())

In [ ]:
df.loc[bad_age, "customer_age"] = np.nan                 # unknown is better than wrong
df.loc[df["discount"] > 1, "discount"] /= 100           # 15 -> 0.15

n = len(df)
df = df[df["quantity"] > 0]                              # can't repair -> remove
log_step("Remove orders with quantity <= 0", n)

### 4.6 Accuracy: cross-field consistency check
`total_amount` **should** equal `quantity × unit_price × (1 − discount) + shipping_cost`. Does it?

In [ ]:
expected = df["quantity"] * df["unit_price"] * (1 - df["discount"]) + df["shipping_cost"]
df["amount_mismatch"] = (expected - df["total_amount"]).abs() > 0.05

print("Orders whose total does not reconcile:", df["amount_mismatch"].sum())
df.loc[df["amount_mismatch"], ["order_id", "quantity", "unit_price", "discount", "shipping_cost", "total_amount"]].head()

💡 We **flag** these rows instead of deleting them. They are suspicious, and we'll investigate them in Session 2 (anomalies!).

### 4.7 Completeness: missing values

📖 **Why is data missing?**
- **MCAR** (completely at random): a sensor glitch. Safe to drop or fill.
- **MAR** (at random, explained by other columns): e.g. age is missing more often for guest checkouts.
- **MNAR** (not at random): the reason relates to the value itself, e.g. unhappy customers don't leave a rating. ⚠️ Filling it in can hide the truth!

| Strategy | When |
|---|---|
| Drop rows | Very few missing values, and they can't be recovered |
| Fill from the **same entity** | e.g. the same customer's age on other orders |
| Fill with median / mode | Numeric / categorical, random missingness |
| Fill with **group** median | When the value depends on a group (e.g. shipping method) |
| `"Unknown"` category | Categorical, where missingness is informative |
| Leave as NaN + add a flag | When the missingness itself means something (MNAR) |

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0] * 100
ax = missing.plot.barh(color="tab:red")
ax.set_xlabel("% missing")
ax.set_title("Missing values per column")
ax.invert_yaxis()
plt.show()

In [ ]:
# 1) order_date: ~1 % missing, can't be guessed -> drop
n = len(df)
df = df.dropna(subset=["order_date"])
log_step("Drop orders without a date", n)

# 2) customer_age: first borrow it from the same customer's other orders, then use the median
df["customer_age"] = df["customer_age"].fillna(df.groupby("customer_id")["customer_age"].transform("median"))
print("Ages still missing after same-customer fill:", df["customer_age"].isna().sum())
df["customer_age"] = df["customer_age"].fillna(df["customer_age"].median()).round()

# 3) delivery_days & shipping_cost depend on shipping_method -> group median
for col in ["delivery_days", "shipping_cost"]:
    df[col] = df[col].fillna(df.groupby("shipping_method")[col].transform("median"))

# 4) payment_method -> explicit "Unknown" category
df["payment_method"] = df["payment_method"].fillna("Unknown")

# 5) customer_rating: MNAR -> keep NaN, add a flag
df["has_rating"] = df["customer_rating"].notna()

df.isna().sum()

### 🧪 Exercise 4.3: Who doesn't rate?

Which **city** has the highest share of orders **without** a rating? (hint: `groupby` + `mean` on `has_rating`)

In [ ]:
# your code here

### 4.8 Final touches, summary & save

In [ ]:
for col in ["city", "category", "shipping_method", "payment_method"]:
    df[col] = df[col].astype("category")
df["quantity"] = df["quantity"].astype(int)
df["delivery_days"] = df["delivery_days"].astype(int)
df = df.reset_index(drop=True)

print(f"Raw  : {raw.shape[0]:,} rows, {raw.isna().sum().sum():,} missing cells")
print(f"Clean: {df.shape[0]:,} rows, {df.isna().sum().sum():,} missing cells (ratings only, on purpose)")
pd.DataFrame(cleaning_log)

In [ ]:
df.to_parquet(DATA / "orders_clean.parquet", index=False)
df.to_csv(DATA / "orders_clean.csv", index=False)

for f in ["orders_raw.csv", "orders_clean.csv", "orders_clean.parquet"]:
    print(f"{f:<22} {(DATA / f).stat().st_size / 1024:>7.0f} KB")

pd.read_parquet(DATA / "orders_clean.parquet").dtypes    # types survived! (CSV would lose them)

---
## 5. 🏆 Challenge: validate against reference data *(⏱ 10 min)*

Real data rarely lives in one table. Use the **product catalog** (`products`) and **customers** (`customers`) you loaded in Section 2.

1. Merge `df` with `products` on `product_id`. Does the order's `category` match the catalog's category for **every** row?
2. Compute `price_ratio = unit_price / list_price`. How many orders are priced more than **2×** the list price? 🤔
3. Merge in `membership` from `customers`. Count orders and total revenue per membership level.

### 🧪 Exercise 5: Challenge

Fill in the blanks (`...`).

In [ ]:
# 1) Category check
chk = df.merge(products[["product_id", "category", "list_price"]], on="product_id", suffixes=("", "_catalog"))
print("Category mismatches:", ...)

# 2) Price check
chk["price_ratio"] = ...
print("Orders priced > 2x list:", ...)

# 3) Membership
chk = chk.merge(customers[["customer_id", "membership"]], on="customer_id", how="left")
...

---
## ✅ Recap: your data-cleaning checklist
- [ ] Read correctly (separator, encoding, decimal, sheet, header rows, nesting)
- [ ] Inspect: `shape`, `info`, `describe`, `value_counts`, `nunique`, `sample`
- [ ] Remove exact duplicates and resolve duplicate IDs
- [ ] Drop useless columns (constant or almost empty)
- [ ] Fix types (numbers, dates, booleans)
- [ ] Standardize categories (trim, case, synonyms)
- [ ] Apply business rules (impossible values)
- [ ] Cross-field checks (do totals add up?)
- [ ] Handle missing values **deliberately** (drop / fill / flag)
- [ ] Save in a typed format (Parquet) and **log your steps**

### ➡️ Next session: Exploratory analysis, correlation, anomaly detection & building a reusable pipeline.